In [ ]:
import os
import glob

# 1. Cerca il file in Google Drive
print("Ricerca del file in Google Drive...")
matches = glob.glob("/content/drive/MyDrive/**/tesi_colab.tar.gz", recursive=True)

if not matches:
    print("❌ File 'tesi_colab.tar.gz' non trovato in Drive.")
    print("Contenuto di MyDrive/uni (se esiste):")
    if os.path.exists("/content/drive/MyDrive/uni"):
        print(os.listdir("/content/drive/MyDrive/uni"))
    else:
        print("La cartella '/content/drive/MyDrive/uni' non esiste. Verifica il nome o se il caricamento su Drive è terminato.")
else:
    archive_path = matches[0]
    print(f"✅ Trovato: {archive_path}")

    # 2. Crea la cartella ed estrai l'archivio
    !mkdir -p /content/tesi
    !tar -xzf "{archive_path}" -C /content/tesi
    print("✅ Estrazione completata con successo!")

# 3. Spostati nella directory di lavoro
if os.path.exists("/content/tesi/SongFormer"):
    %cd /content/tesi/SongFormer
    print(f"Cartella corrente: {os.getcwd()}")


Ricerca del file in Google Drive...
❌ File 'tesi_colab.tar.gz' non trovato in Drive.
Contenuto di MyDrive/uni (se esiste):
La cartella '/content/drive/MyDrive/uni' non esiste. Verifica il nome o se il caricamento su Drive è terminato.


In [ ]:
import re, subprocess, sys
skip = {"torch","torchaudio","triton","numpy","pandas","scipy","scikit-learn","pillow","pyarrow",
        "wandb","gradio","gpustat","tensorboard","pypdf2","blessed","pesq","msaf","jams","vmo"}
pk = [re.split(r"[=<>!~ ]", l.split("#")[0].strip())[0]
      for l in open("/content/tesi/SongFormer/requirements.txt") if l.split("#")[0].strip()]
pk = [p for p in pk if p.lower() not in skip]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pk])

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'lightning', 'transformers', 'accelerate', 'datasets', 'tokenizers', 'huggingface-hub', 'safetensors', 'librosa', 'audioread', 'soundfile', 'auraloss', 'nnAudio', 'julius', 'soxr', 'mir_eval', 'matplotlib', 'seaborn', 'hydra-core', 'omegaconf', 'fire', 'click', 'einops', 'einx', 'x-transformers', 'x-clip', 'ema-pytorch', 'schedulefree', 'torchmetrics', 'h5py', 'ftfy', 'regex', 'pypinyin', 'textgrid', 'pylrc', 'modelscope', 'tqdm', 'loguru', 'joblib', 'easydict', 'addict', 'beartype', 'muq', 'einops', 'beartype'], returncode=0)

In [ ]:
# Idea: MuQ's attention builds the full (frames x frames) score matrix TWICE (scores + softmax). That is the 5.5 GiB
# allocation that fails. PyTorch's scaled_dot_product_attention computes the SAME attention without that matrix,
# in the same fp32 precision, so the paper's numerics are kept and only memory drops.
# The patch self-tests against the original attention at start-up and switches itself off if they don't match.

# ---------------------------------------------------------------
# CELL 1: apply the patch to infer.py (backup: infer.py.bak2)
# ---------------------------------------------------------------
P = "/content/tesi/SongFormer/src/SongFormer/infer/infer.py"

BLOCK = '''# --- SDPA_PATCH: same attention math, O(n) memory instead of O(n^2) ---
import torch.nn.functional as _F
from transformers.models.wav2vec2_conformer import modeling_wav2vec2_conformer as _w2v
_A = _w2v.Wav2Vec2ConformerSelfAttention
_orig_fwd = _A.forward


def _sdpa_forward(self, hidden_states, attention_mask=None,
                  relative_position_embeddings=None, output_attentions=False):
    if self.position_embeddings_type not in ("rotary", None) or output_attentions:
        return _orig_fwd(self, hidden_states, attention_mask, relative_position_embeddings, output_attentions)
    b = hidden_states.size(0)
    qk = hidden_states
    if self.position_embeddings_type == "rotary":
        qk = self._apply_rotary_embedding(qk, relative_position_embeddings)
    q = self.linear_q(qk).view(b, -1, self.num_heads, self.head_size).transpose(1, 2)
    k = self.linear_k(qk).view(b, -1, self.num_heads, self.head_size).transpose(1, 2)
    v = self.linear_v(hidden_states).view(b, -1, self.num_heads, self.head_size).transpose(1, 2)
    o = _F.scaled_dot_product_attention(
        q, k, v, attn_mask=attention_mask, dropout_p=self.dropout.p if self.training else 0.0)
    o = o.transpose(1, 2).reshape(b, -1, self.num_heads * self.head_size)
    return self.linear_out(o), None


def _sdpa_selftest():
    import types
    cfg = types.SimpleNamespace(hidden_size=64, num_attention_heads=4,
                                position_embeddings_type="rotary", attention_dropout=0.0)
    m = _A(cfg).eval()
    x = torch.randn(1, 50, 64)
    ang = torch.randn(50, 16)
    emb = torch.stack([ang.cos(), ang.sin()])[:, :, None, None, :]      # [2, T, 1, 1, head_size]
    with torch.no_grad():
        ref = _orig_fwd(m, x, None, emb, False)[0]
        new = _sdpa_forward(m, x, None, emb, False)[0]
    return torch.allclose(ref, new, atol=1e-5, rtol=1e-4)


try:
    if _sdpa_selftest():
        _A.forward = _sdpa_forward
        print("SDPA attention patch ACTIVE (self-test passed)")
    else:
        print("SDPA attention patch NOT used: self-test mismatch")
except Exception as _e:
    print("SDPA attention patch NOT used:", repr(_e))
# --- end SDPA_PATCH ---

'''

ANCHOR = 'mp.set_start_method("spawn", force=True)'
s = open(P).read()
assert "SDPA_PATCH" not in s, "already patched"
assert s.count(ANCHOR) == 1, "anchor line not found exactly once - paste me your infer.py header"
open(P + ".bak2", "w").write(s)
open(P, "w").write(s.replace(ANCHOR, BLOCK + ANCHOR))
print("patched")

patched


In [ ]:
# write new infer.sh
%%writefile /content/tesi/SongFormer/src/SongFormer/infer.sh
export CUDA_VISIBLE_DEVICES=0
echo "use gpu ${CUDA_VISIBLE_DEVICES}"

export PYTHONPATH=.:../third_party:$PYTHONPATH
export PYTHONUNBUFFERED=1

export OMP_NUM_THREADS=1
export MPI_NUM_THREADS=1
export NCCL_P2P_DISABLE=1
export NCCL_IB_DISABLE=1

python infer/infer.py \
-i /content/tesi/SongFormer/songs.scp \
-o /content/tesi/SongFormer/results_patched \
--model SongFormer \
--checkpoint SongFormer.safetensors \
--config_path SongFormer.yaml \
-gn 1 \
-tn 1 \
$DEBUG
# --no_rule_post_processing

Overwriting /content/tesi/SongFormer/src/SongFormer/infer.sh


In [ ]:
import os, pandas as pd
ROOT = "/content/tesi/SongFormer"; SF = f"{ROOT}/src/SongFormer"

# 1. SCP: one absolute audio path per line
df = pd.read_csv("/content/tesi/sampled_eval_dataset_enriched.csv")
paths = [os.path.abspath(f"{ROOT}/downloaded_data/{p}") for p in df.audio_path]
paths = [p for p in paths if os.path.exists(p)]
print(paths)
open(f"{ROOT}/songs.scp", "w").write("\n".join(paths) + "\n")
print(len(paths), "songs")

['/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0928_takeabow.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0450_more.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0851_openurmind.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0112_giveitup.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0770_kingofmycastle.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0415_illbewaiting.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0328_youmakemefeeldc.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0930_takeyourtime.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0866_pokerfacedave.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0592_blameit.wav', '/content/tesi/SongFormer/downloaded_data/data/audios/HarmonixSet/BHX_0474_p

In [ ]:
!cd {ROOT}/src/SongFormer && bash infer.sh
print(os.listdir(f"{ROOT}/results_patched"))

use gpu 0
2026-09-20 13:35:07.606867: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
SDPA attention patch ACTIVE (self-test passed)
num_threads: 0 on GPU 0
add data to queue: 100% 50/50 [00:00<00:00, 119291.92it/s]
getting inference output:   0% 0/50 [00:00<?, ?it/s]SDPA attention patch ACTIVE (self-test passed)
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
2026-09-20 13:35:31.602 | INFO     | __mp_main__:inference:191 - Loading EMA model parameters
2026-09-20 13:35:31.653 | INFO     | __mp_main__:inference:193 - EMA initialized
2026-09-20 13:35:31.685 | INFO     |

# we run eval

first on processed_results.jsonl

and then on processed_results_patched.jsonl - fp16_songs.txt

SongFormer outputs can be converted using the utility script src/SongFormer/utils/convert_res2msa_txt.py

Input folder: /content/tesi/SongFormer/results
Output folder: /content/tesi/SongFormer/converted_eval_results
--------------------------------------------------
Found 44 JSON files
✓ Processed: BHX_0799_lovesexmagic.json -> BHX_0799_lovesexmagic.txt
✓ Processed: BHX_0112_giveitup.json -> BHX_0112_giveitup.txt
✓ Processed: BHX_0197_nothinonyou.json -> BHX_0197_nothinonyou.txt
✓ Processed: BHX_0491_skyscraper.json -> BHX_0491_skyscraper.txt
✓ Processed: BHX_0450_more.json -> BHX_0450_more.txt
✓ Processed: BHX_0392_givemeyourhand.json -> BHX_0392_givemeyourhand.txt
✓ Processed: BHX_0770_kingofmycastle.json -> BHX_0770_kingofmycastle.txt
✓ Processed: BHX_0592_blameit.json -> BHX_0592_blameit.txt
✓ Processed: BHX_0807_makemebetter.json -> BHX_0807_makemebetter.txt
✓ Processed: BHX_0248_screamingfor.json -> BHX_0248_screamingfor.txt
✓ Processed: BHX_0668_fallinganthem.json -> BHX_0668_fallinganthem.txt
✓ Processed: BHX_0811_miami2ibiza.json -> BHX_0811_miami2ibiza.txt
✓ Processed: BHX_0748_i

In [ ]:
#get ground truth from google drive
ground_truth_folder = "/content/drive/MyDrive/uni /tesi/HarmonixSet"

#check that exists

os.path.exists(ground_truth_folder)

True

In [ ]:
#patch eval.sh
%%writefile /content/tesi/SongFormer/src/SongFormer/eval.sh
#!/bin/bash
set -e

if [ "$#" -lt 3 ]; then
    echo "Error: Missing required arguments."
    echo "Usage: $0 <EST_DIR> <ANN_DIR> <OUTPUT_DIR>"
    exit 1
fi

EST_DIR="$1"
ANN_DIR="$2"
OUTPUT_DIR="$3"

# Resolve directory of this script (/content/tesi/SongFormer/src/SongFormer)
SCRIPT_DIR="$(cd "$(dirname "${BASH_SOURCE[0]}")" &>/dev/null && pwd)"

export CUDA_VISIBLE_DEVICES=-1
export PYTHONPATH="$SCRIPT_DIR:$PYTHONPATH"

export HYDRA_FULL_ERROR=1
export OMP_NUM_THREADS=1
export MPI_NUM_THREADS=1
export NCCL_P2P_DISABLE=1
export NCCL_IB_DISABLE=1

echo "Evaluating: $EST_DIR against $ANN_DIR"
echo "Output directory: $OUTPUT_DIR"
mkdir -p "$OUTPUT_DIR"

python "$SCRIPT_DIR/evaluation/eval_infer_results.py" \
    --ann_dir "$ANN_DIR" \
    --est_dir "$EST_DIR" \
    --output_dir "$OUTPUT_DIR" \
    --prechorus2what verse

Overwriting /content/tesi/SongFormer/src/SongFormer/eval.sh


In [ ]:
# run eval for unpatched
EST_DIR = "/content/tesi/SongFormer/converted_eval_results"
ANN_DIR = ground_truth_folder
OUTPUT_DIR = "/content/tesi/SongFormer/unpatched_evaluation_output"

# Run evaluation
!bash /content/tesi/SongFormer/src/SongFormer/eval.sh "$EST_DIR" "$ANN_DIR" "$OUTPUT_DIR"

Evaluating: /content/tesi/SongFormer/converted_eval_results against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/tesi/SongFormer/unpatched_evaluation_output
2026-09-20 14:34:31.434 | INFO     | __main__:main:96 - Common number of files: 44
  0% 0/44 [00:00<?, ?it/s]2026-09-20 14:34:31.436 | INFO     | __main__:main:103 - Processing BHX_0273_technologic.txt
  2% 1/44 [00:00<00:20,  2.07it/s]2026-09-20 14:34:31.918 | INFO     | __main__:main:103 - Processing BHX_0332_alejandro.txt
  5% 2/44 [00:00<00:20,  2.04it/s]2026-09-20 14:34:32.415 | INFO     | __main__:main:103 - Processing BHX_0770_kingofmycastle.txt
  7% 3/44 [00:01<00:20,  2.03it/s]2026-09-20 14:34:32.907 | INFO     | __main__:main:103 - Processing BHX_0676_figure8.txt
  9% 4/44 [00:01<00:19,  2.04it/s]2026-09-20 14:34:33.394 | INFO     | __main__:main:103 - Processing BHX_0668_fallinganthem.txt
 11% 5/44 [00:02<00:18,  2.06it/s]2026-09-20 14:34:33.870 | INFO     | __main__:main:103 - Processing BHX_0

In [ ]:
# run eval for patched - fp16
EST_DIR = "/content/tesi/SongFormer/converted_eval_results_patched"
ANN_DIR = ground_truth_folder
OUTPUT_DIR = "/content/tesi/SongFormer/patched_evaluation_output"

# Read fp16 songs into a set for efficient lookup
fp16_songs = set()
fp16_songs_file = "/content/tesi/SongFormer/fp16_songs.txt"

if os.path.exists(fp16_songs_file):
    with open(fp16_songs_file, 'r') as f:
        for line in f:
            fp16_songs.add(line.strip())

# Remove songs that appear in fp16_songs from EST_DIR
print(f"Checking {EST_DIR} for songs to remove based on {fp16_songs_file}")
removed_count = 0
if os.path.exists(EST_DIR):
    for filename in os.listdir(EST_DIR):
        if filename.endswith(".txt"):
            song_id = os.path.splitext(filename)[0]
            if song_id in fp16_songs:
                file_to_remove = os.path.join(EST_DIR, filename)
                os.remove(file_to_remove)
                removed_count += 1
                # print(f"Removed: {filename}") # Uncomment for verbose logging

print(f"Removed {removed_count} evaluation result files from {EST_DIR}")

# Run evaluation
!bash /content/tesi/SongFormer/src/SongFormer/eval.sh "$EST_DIR" "$ANN_DIR" "$OUTPUT_DIR"


Checking /content/tesi/SongFormer/converted_eval_results_patched for songs to remove based on /content/tesi/SongFormer/fp16_songs.txt
Removed 6 evaluation result files from /content/tesi/SongFormer/converted_eval_results_patched
Evaluating: /content/tesi/SongFormer/converted_eval_results_patched against /content/drive/MyDrive/uni /tesi/HarmonixSet
Output directory: /content/tesi/SongFormer/patched_evaluation_output
2026-09-20 14:41:28.060 | INFO     | __main__:main:96 - Common number of files: 44
  0% 0/44 [00:00<?, ?it/s]2026-09-20 14:41:28.061 | INFO     | __main__:main:103 - Processing BHX_0273_technologic.txt
2026-09-20 14:41:28.082 | INFO     | __main__:main:103 - Processing BHX_0763_iwasafool.txt
2026-09-20 14:41:28.112 | INFO     | __main__:main:103 - Processing BHX_0248_screamingfor.txt
2026-09-20 14:41:28.155 | INFO     | __main__:main:103 - Processing BHX_0770_kingofmycastle.txt
  9% 4/44 [00:00<00:01, 32.55it/s]2026-09-20 14:41:28.184 | INFO     | __main__:main:103 - Process

In [ ]:
import os

unpatched_output_dir = "/content/tesi/SongFormer/unpatched_evaluation_output"
patched_output_dir = "/content/tesi/SongFormer/patched_evaluation_output"

def display_results(directory, name):
    print(f"\n--- {name} Results ({directory}) ---")
    results_file_found = False
    summary_files_to_check = ["eval_infer_summary.md", "eval_infer_summary.csv", "eval_infer.csv"]

    if not os.path.exists(directory):
        print(f"Error: Directory {directory} does not exist.")
        return

    for summary_filename in summary_files_to_check:
        summary_file_path = os.path.join(directory, summary_filename)
        if os.path.exists(summary_file_path):
            print(f"Found and displaying: {summary_filename}")
            try:
                with open(summary_file_path, 'r') as f:
                    print(f.read())
                results_file_found = True
                break
            except Exception as e:
                print(f"Error reading {summary_file_path}: {e}")

    if not results_file_found:
        print(f"No recognized summary file found (checked: {', '.join(summary_files_to_check)}). ")
        print(f"Directory contents: {os.listdir(directory)}")


# Display unpatched results
display_results(unpatched_output_dir, "Unpatched Evaluation")

# Display patched results
display_results(patched_output_dir, "Patched Evaluation (FP16 filtered)")



--- Unpatched Evaluation Results (/content/tesi/SongFormer/unpatched_evaluation_output) ---
Found and displaying: eval_infer_summary.md
|    |   num_samples |    HR.5F |    HR3F |     HR1F |      PWF |      Sf |     acc |     iou |   iou_bridge |   iou_chorus |   iou_verse |   iou_silence |   iou_outro |   iou_inst |   iou_intro |
|---:|--------------:|---------:|--------:|---------:|---------:|--------:|--------:|--------:|-------------:|-------------:|------------:|--------------:|------------:|-----------:|------------:|
|  0 |            44 | 0.703555 | 0.81178 | 0.765309 | 0.770852 | 0.79657 | 0.77362 | 0.62325 |     0.451791 |     0.677395 |    0.754831 |       0.52242 |     0.33876 |   0.367499 |    0.802447 |


--- Patched Evaluation (FP16 filtered) Results (/content/tesi/SongFormer/patched_evaluation_output) ---
Found and displaying: eval_infer_summary.md
|    |   num_samples |    HR.5F |    HR3F |     HR1F |      PWF |      Sf |     acc |     iou |   iou_verse |   iou_outro 

In [ ]:
import pandas as pd
import os

unpatched_eval_csv = os.path.join(unpatched_output_dir, 'eval_infer.csv')
patched_eval_csv = os.path.join(patched_output_dir, 'eval_infer.csv')

print(f"Loading unpatched results from: {unpatched_eval_csv}")
print(f"Loading patched results from: {patched_eval_csv}")

try:
    # Load CSVs, treating the first column as the index (likely the song ID)
    df_unpatched = pd.read_csv(unpatched_eval_csv, index_col=0)
    df_patched = pd.read_csv(patched_eval_csv, index_col=0)

    # Ensure both dataframes have the same set of columns before comparison
    common_cols = list(set(df_unpatched.columns) & set(df_patched.columns))
    df_unpatched = df_unpatched[common_cols].sort_index(axis=1)
    df_patched = df_patched[common_cols].sort_index(axis=1)

    # Align the dataframes by index (song_id) before comparison
    # This handles cases where one might have songs the other doesn't (though in this case it should be the same 44)
    common_index = list(set(df_unpatched.index) & set(df_patched.index))
    df_unpatched_aligned = df_unpatched.loc[common_index].sort_index()
    df_patched_aligned = df_patched.loc[common_index].sort_index()


    if df_unpatched_aligned.equals(df_patched_aligned):
        print("\nBoth unpatched and patched eval_infer.csv files (for common songs) are identical.")
    else:
        print("\nDifferences found between unpatched and patched eval_infer.csv files (for common songs).")
        # Find rows that are different
        diff_df = df_unpatched_aligned.compare(df_patched_aligned, align_axis=1, keep_shape=True)
        print("\nDetailed differences (unpatched vs patched):")
        display(diff_df)

except FileNotFoundError:
    print("Error: One or both 'eval_infer.csv' files not found. Please ensure evaluation ran successfully.")
except Exception as e:
    print(f"An error occurred during comparison: {e}")


Loading unpatched results from: /content/tesi/SongFormer/unpatched_evaluation_output/eval_infer.csv
Loading patched results from: /content/tesi/SongFormer/patched_evaluation_output/eval_infer.csv

Differences found between unpatched and patched eval_infer.csv files (for common songs).

Detailed differences (unpatched vs patched):


D              DevE2R              DevR2E            \
                self     other      self     other      self     other   
HitRate_3P                                                               
0.368421         NaN       NaN       NaN       NaN       NaN       NaN   
0.533333         NaN       NaN       NaN       NaN       NaN       NaN   
0.538462         NaN       NaN       NaN       NaN       NaN       NaN   
0.545455         NaN       NaN       NaN       NaN       NaN       NaN   
0.550000         NaN       NaN       NaN       NaN       NaN       NaN   
0.562500         NaN       NaN       NaN       NaN       NaN       NaN   
0.588235         NaN       NaN       NaN       NaN       NaN       NaN   
0.588235         NaN       NaN       NaN       NaN       NaN       NaN   
0.615385         NaN       NaN       NaN       NaN       NaN       NaN   
0.625000         NaN       NaN       NaN       NaN       NaN       NaN   
0.636364         NaN       NaN       NaN       NaN       NaN       NaN   
0.687500         NaN       NaN       NaN       NaN       NaN       NaN   
0.687500         NaN       NaN       NaN       NaN       NaN       NaN   
0.692308         NaN       NaN       NaN       NaN       NaN       NaN   
0.714286         NaN       NaN       NaN       NaN       NaN       NaN   
0.750000         NaN       NaN       NaN       NaN       NaN       NaN   
0.750000         NaN       NaN       NaN       NaN       NaN       NaN   
0.750000         NaN       NaN       NaN       NaN       NaN       NaN   
0.750000         NaN       NaN       NaN       NaN       NaN       NaN   
0.769231         NaN       NaN       NaN       NaN       NaN       NaN   
0.785714    0.568538  0.685142  0.294045  0.037795  0.224910  0.034000   
0.785714    0.685142  0.568538  0.037795  0.294045  0.034000  0.224910   
0.800000         NaN       NaN       NaN       NaN       NaN       NaN   
0.812500    0.748978  0.712845  0.054525  0.090060  0.039290  0.084695   
0.812500    0.712845  0.748978  0.090060  0.054525  0.084695  0.039290   
0.812500         NaN       NaN       NaN       NaN       NaN       NaN   
0.818182         NaN       NaN       NaN       NaN       NaN       NaN   
0.818182         NaN       NaN       NaN       NaN       NaN       NaN   
0.833333         NaN       NaN       NaN       NaN       NaN       NaN   
0.846154    0.564704  0.820958  0.216690  0.028790  0.216690  0.027070   
0.846154         NaN       NaN       NaN       NaN       NaN       NaN   
0.846154    0.820958  0.564704  0.028790  0.216690  0.027070  0.216690   
0.866667         NaN       NaN       NaN       NaN       NaN       NaN   
0.875000         NaN       NaN       NaN       NaN       NaN       NaN   
0.882353    0.814939  0.767860  0.190870  0.404680  0.157920  0.405145   
0.882353    0.767860  0.814939  0.404680  0.190870  0.405145  0.157920   
0.900000         NaN       NaN       NaN       NaN       NaN       NaN   
0.909091         NaN       NaN       NaN       NaN       NaN       NaN   
0.928571         NaN       NaN       NaN       NaN       NaN       NaN   
0.937500         NaN       NaN       NaN       NaN       NaN       NaN   
1.000000    0.749711  0.797700  0.136190  0.285170  0.136190  0.285170   
1.000000    0.880380  0.886905  0.114370  0.099050  0.114370  0.099960   
1.000000    0.886905  0.749711  0.099050  0.136190  0.099960  0.136190   
1.000000    0.797700  0.880380  0.285170  0.114370  0.285170  0.114370   

             DevtE2R             DevtR2E            ...  iou-inst            \
                self     other      self     other  ...      self     other   
HitRate_3P                                          ...                       
0.368421         NaN       NaN       NaN       NaN  ...       NaN       NaN   
0.533333         NaN       NaN       NaN       NaN  ...       NaN       NaN   
0.538462         NaN       NaN       NaN       NaN  ...       NaN       NaN   
0.545455         NaN       NaN       NaN       NaN  ...       NaN 

the results are the same!

so can we keep the other 6? we'll find about later. anyways we will flag them for completeness